# m02_l03_json

Exobean Engineering. Run the cells from top to bottom.

In [ ]:
# Finds the code companion. On Google Colab it downloads it once.
import glob, io, os, sys, urllib.request, zipfile
URL = "https://github.com/exobean/ai-agents-level1/releases/latest/download/shopassist-companion.zip"
if not os.path.isdir("shopassist"):
    if os.path.isdir(os.path.join("..", "shopassist")):
        os.chdir("..")  # opened from the notebooks folder on your computer
    else:
        found = glob.glob("*-companion")  # kept: your edited answers stay
        if not found and "google.colab" in sys.modules:
            try:
                data = io.BytesIO(urllib.request.urlopen(URL).read())
            except OSError:  # network blocks the download: upload the zip
                from google.colab import files
                data = next(iter(files.upload()))
            zipfile.ZipFile(data).extractall(".")
            found = glob.glob("*-companion")
        if found:
            os.chdir(found[0])
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("Companion ready" if os.path.isdir("shopassist") else
      "Companion not found: see Lesson 0.1 of the book")


## Json

In [ ]:
import json

from shopassist.client import post_json
from shopassist.mockserver import MockProvider
from shopassist.structured import check, extract_json

body = {"model": "mock", "max_tokens": 200,
        "system": "Read the customer message. Reply with JSON only.",
        "messages": [{"role": "user", "content": "Where's #1042?!"}]}

with MockProvider() as server:
    text = post_json(server.url, body, "test-key")["content"][0]["text"]
print("raw reply:", repr(text[:38]), "...")

try:
    json.loads(text)                      # the naive way
except json.JSONDecodeError as err:
    print("json.loads failed:", err.msg)

data = extract_json(text)                 # the careful way
print("extracted:", data)
print("problems: ", check(data, {"order_id": str, "intent": str}))
